Programatic solutions to the 3 big problems in HMMs. Credit to Rabiner as well as Jurafsky and Martin. 

In [3]:
import numpy as np

Initialising ice-cream Markov chain found in "Speech and Language Processing" by Jurafsky and Martin.

In [9]:
# cold - hot
A = np.array([[0.5, 0.5], [0.4, 0.6]])
print(A)
B = np.array([[0.5, 0.4, 0.1], [0.2, 0.4, 0.4]])
print(B)
pi = [0.2, 0.8]
O=[2,0,2]

[[0.5 0.5]
 [0.4 0.6]]
[[0.5 0.4 0.1]
 [0.2 0.4 0.4]]


In [10]:
A @ B
c = np.array([2, 1, 0])
c * B

array([[1. , 0.4, 0. ],
       [0.4, 0.4, 0. ]])

# The Forward Algorithm
A dynamic programming solution to the problem:

Given a Hiden Markov Model $\lambda = (A,B)$ and an observation sequence $O$, determine the likelihood $P(O|\lambda)$.

In [247]:
def forward(A, B, pi, O):
    """
    A: ndarray(N, N) state transition probabilities
    B: ndarray(N, M) state observation probabilities
    pi: list[N] first state probabilities
    O: list[T] enumerated observation sequence
    """

    N = A.shape[0]
    T = len(O)
    pi = np.asarray(pi)
    alpha = np.zeros((N,T))

    # Initialise the first column of alpha; each ith row is the probability
    # of starting in state i and observing the first observation (B[0]) there.
    alpha[:, 0] = pi * B[:, O[0]]

    for t in range(1, T):
        # each row of A to be pairwise multiplied by the row vector of B_t
        # so each (i,j) in i_j_ot is the probability of moving from state i
        # to j and then observing O[t] at j.
        # shape: (N,N)
        i_j_ot = A * B[:, O[t]] 

        # in alpha[:, t-1], the ith row is the probability of being in 
        # state 0 at time t-1 given the observation sequence up to t-1. 
        # Thus, matmul with i_j_ot 
        alpha[:, t] = alpha[:, t-1] @ i_j_ot

    return alpha

def forward_prob(A, B, pi, O):
    return forward(A, B, pi, O)[:, len(O)-1].sum()

In [248]:
forward_prob(A,B,pi,O)

np.float64(0.028562000000000008)

# Viterbi
A dynamic programming algorithm to find an optimal solution to the problem:

Given as input a Hidden Markov model $\lambda=(A,B)$ and a sequence of observations $O=o_1,o_2,...o_T$, find the most probable sequence of states $Q=q_1q_2q_3...q_T$.

In [33]:
def viterbi(A, B, pi, O):
    """
    A: ndarray(N, N) state transition probabilities
    B: ndarray(N, M) state observation probabilities
    pi: list[N] first state probabilities
    O: list[T] enumerated observation sequence
    """

    N = A.shape[0]
    T = len(O)
    pi = np.asarray(pi)

    # every v[j, t] is the maximum probability over all state sequences
    # from 1 to t-1 of achieving said state sequence and the observation 
    # sequence from 1 to t and being in state j at time t given the model.
    v = np.zeros((N,T))
    bt = np.zeros((N,T), dtype=int)

    v[:, 0] = pi * B[:, O[0]]
    bt[:, 0] = 0

    for t in range(1, T):
        i_j_ot = A * B[:, O[t]] # shape: (n,n)
        v_last = v[:, t-1].reshape(-1,1) # shape: (n,1)

        # each entry of v_last is the probability of being in that state at t-1
        # the reshape ensures that each (i, j) in v_last * i_j_ot
        # gives the probability of being in state j from state i
        # axis=0 takes the most likely previous state from destination
        v[:, t] = np.max(v_last * i_j_ot, axis=0)
        bt[:, t] = np.argmax(v_last * i_j_ot, axis=0)

        # for s in range(0, N):
        #     # RHS: max probability of moving from any state to state s
        #     v[s, t] = np.max(v_last * i_j_ot[:, s])
        #     bt[s, t] = np.argmax(v_last * i_j_ot[:, s])


    # most likely final state at time T
    bestpathpointer = np.argmax(v[:, T-1])
    bestpath = [None] * T
    bestpath[T-1] = bestpathpointer
    for i in range(1, T):
        # bt[s, t] is the most likely state at t-1 which led to state s at time t
        bestpath[T-1-i] = bt[bestpath[T-i], T-i]

    bestpathprobability = v[:, T-1].max()

    print(v)

    return bestpath, bestpathprobability

In [34]:
bp, bpp = viterbi(A, B, pi, O)
print(bp)
print(bpp)

[[0.02   0.064  0.0032]
 [0.32   0.0384 0.0128]]
[np.int64(1), np.int64(0), np.int64(1)]
0.012800000000000004


In [32]:
bp2, bpp2 = viterbi(A, B, pi, O)
print(bp2)
print(bpp2)

[[0.005  0.002 ]
 [0.064  0.0384]]
[[0.0032   0.0128  ]
 [0.001536 0.009216]]
[[0.02   0.064  0.0032]
 [0.32   0.0384 0.0128]]
[np.int64(1), np.int64(0), np.int64(1)]
0.012800000000000004


In [30]:
c = np.array([1, 2, 1])
d = np.array([[1, 1, 0], [2, 0, 2], [0, 3, 3]])
print(c * d)
np.max(c*d, axis=0)

[[1 2 0]
 [2 0 2]
 [0 6 3]]


array([2, 6, 3])

In [7]:
n = np.array([[1, 0], [2, 3]])
np.max(n, axis=1)

array([1, 3])